# Workshop 6 Live Coding: Functions & Scope

This notebook covers:
- `FunctionDefinition`
- `FunctionCall`
- local environments
- parent lookup
- parameter binding
- nested calls

In [1]:
from dataclasses import dataclass
from typing import Any

In [2]:
@dataclass
class Number:
    value: int

@dataclass
class Variable:
    name: str

@dataclass
class Assignment:
    name: str
    value: Any

@dataclass
class BinaryOp:
    op: str
    left: Any
    right: Any

@dataclass
class FunctionDefinition:
    name: str
    parameters: list[str]
    body: Any

@dataclass
class FunctionCall:
    name: str
    arguments: list[Any]

# Part 1 — Starter Environment

In [3]:
class EnvironmentStarter:
    def __init__(self):
        self.values = {}
        self.functions = {}

    def define(self, name, value):
        self.values[name] = value
        return value

    def get(self, name):
        if name in self.values:
            return self.values[name]
        raise NameError(f"{name!r} is not defined")

## Solution: Parent-Aware Environment

Live-code:
1. `parent`
2. local lookup
3. parent lookup
4. function storage

In [ ]:
class Environment:
    def __init__(self, parent=None):
        self.values = {}
        self.functions = {}
        self.parent = parent

    def define(self, name, value):
        self.values[name] = value
        return value

    def get(self, name):
        if name in self.values:
            return self.values[name]
        if self.parent is not None:
            return self.parent.get(name)
        raise NameError(f"{name!r} is not defined")

    def define_function(self, name, function):
        self.functions[name] = function

    def get_function(self, name):
        if name in self.functions:
            return self.functions[name]
        if self.parent is not None:
            return self.parent.get_function(name)
        raise NameError(f"Function {name!r} is not defined")

    def __repr__(self):
        return (
            f"Environment(values={self.values}, "
            f"functions={list(self.functions)}, "
            f"has_parent={self.parent is not None})"
        )

# Part 2 — Starter Evaluator

In [ ]:
def evaluate_starter(node, env):
    if isinstance(node, Number):
        return node.value

    if isinstance(node, Variable):
        return env.get(node.name)

    if isinstance(node, Assignment):
        value = evaluate_starter(node.value, env)
        env.define(node.name, value)
        return value

    if isinstance(node, BinaryOp):
        left = evaluate_starter(node.left, env)
        right = evaluate_starter(node.right, env)

        if node.op == "+":
            return left + right
        if node.op == "-":
            return left - right
        if node.op == "*":
            return left * right
        if node.op == "/":
            return left / right

        raise ValueError(f"Unknown operator: {node.op}")

    # TODO: FunctionDefinition
    # TODO: FunctionCall

    raise TypeError(f"Unknown node: {node}")

# Part 3 — Live Coding: Function Definition

In [ ]:
# Add this case before the final TypeError:

# if isinstance(node, FunctionDefinition):
#     env.define_function(node.name, node)
#     return None

# Part 4 — Live Coding: Function Call

In [ ]:
# Add this case before the final TypeError:

# if isinstance(node, FunctionCall):
#     function = env.get_function(node.name)
#
#     argument_values = [
#         evaluate(node_arg, env)
#         for node_arg in node.arguments
#     ]
#
#     local_env = Environment(parent=env)
#
#     for parameter, value in zip(
#         function.parameters,
#         argument_values,
#     ):
#         local_env.define(parameter, value)
#
#     return evaluate(function.body, local_env)

# Part 5 — Complete Solution

In [ ]:
def evaluate(node, env):
    if isinstance(node, Number):
        return node.value

    if isinstance(node, Variable):
        return env.get(node.name)

    if isinstance(node, Assignment):
        value = evaluate(node.value, env)
        env.define(node.name, value)
        return value

    if isinstance(node, BinaryOp):
        left = evaluate(node.left, env)
        right = evaluate(node.right, env)

        if node.op == "+":
            return left + right
        if node.op == "-":
            return left - right
        if node.op == "*":
            return left * right
        if node.op == "/":
            return left / right

        raise ValueError(f"Unknown operator: {node.op}")

    if isinstance(node, FunctionDefinition):
        env.define_function(node.name, node)
        return None

    if isinstance(node, FunctionCall):
        function = env.get_function(node.name)

        if len(function.parameters) != len(node.arguments):
            raise TypeError(
                f"{node.name} expected {len(function.parameters)} "
                f"arguments, got {len(node.arguments)}"
            )

        argument_values = [
            evaluate(argument, env)
            for argument in node.arguments
        ]

        local_env = Environment(parent=env)

        for parameter, value in zip(
            function.parameters,
            argument_values,
        ):
            local_env.define(parameter, value)

        print("local before body:", local_env)
        result = evaluate(function.body, local_env)
        print("local after body: ", local_env)

        return result

    raise TypeError(f"Unknown node: {node}")

# Part 6 — Demo: square(5)

In [ ]:
global_env = Environment()

square = FunctionDefinition(
    name="square",
    parameters=["x"],
    body=BinaryOp("*", Variable("x"), Variable("x")),
)

evaluate(square, global_env)

result = evaluate(
    FunctionCall("square", [Number(5)]),
    global_env,
)

print("result:", result)
print("global:", global_env)

# Part 7 — Demo: Global vs Local Scope

In [ ]:
global_env = Environment()
global_env.define("x", 100)

evaluate(
    FunctionDefinition(
        name="square",
        parameters=["x"],
        body=BinaryOp("*", Variable("x"), Variable("x")),
    ),
    global_env,
)

print("global before:", global_env)

result = evaluate(
    FunctionCall("square", [Number(5)]),
    global_env,
)

print("result:", result)
print("global after:", global_env)
print("global x:", global_env.get("x"))

# Part 8 — Demo: Parent Lookup

In [ ]:
global_env = Environment()
global_env.define("multiplier", 10)

evaluate(
    FunctionDefinition(
        name="scale",
        parameters=["x"],
        body=BinaryOp(
            "*",
            Variable("x"),
            Variable("multiplier"),
        ),
    ),
    global_env,
)

print(
    evaluate(
        FunctionCall("scale", [Number(7)]),
        global_env,
    )
)

# Part 9 — Demo: Multiple Parameters

In [ ]:
global_env = Environment()

evaluate(
    FunctionDefinition(
        name="area",
        parameters=["width", "height"],
        body=BinaryOp(
            "*",
            Variable("width"),
            Variable("height"),
        ),
    ),
    global_env,
)

print(
    evaluate(
        FunctionCall(
            "area",
            [Number(4), Number(5)],
        ),
        global_env,
    )
)

# Part 10 — Demo: Nested Calls

In [ ]:
global_env = Environment()

evaluate(
    FunctionDefinition(
        name="square",
        parameters=["x"],
        body=BinaryOp("*", Variable("x"), Variable("x")),
    ),
    global_env,
)

evaluate(
    FunctionDefinition(
        name="double",
        parameters=["x"],
        body=BinaryOp("*", Variable("x"), Number(2)),
    ),
    global_env,
)

nested = FunctionCall(
    "double",
    [
        FunctionCall(
            "square",
            [Number(5)],
        )
    ],
)

print(evaluate(nested, global_env))

# Key Takeaways

Function definition:
- stores code
- does not execute the body

Function call:
- finds the function
- evaluates arguments
- creates a local environment
- binds parameters
- evaluates the body
- returns the result

Scope:
- look locally first
- then ask the parent